# Lab 19: Fourier — Signals, Waves, and Hidden Frequencies

This lab is a full computational companion to Chapter 19.

The goal is not only to run `np.fft.fft`. The goal is to understand Fourier analysis as linear algebra:

- a signal is a vector;
- waves are basis directions;
- Fourier coefficients are coordinates;
- filtering is editing coordinates;
- compression is keeping the most important coordinates.

We will move from simple one-dimensional signals to noisy signals, compression, spectrogram-style thinking, images, and a final high-dimensional viewpoint.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(19)
plt.rcParams['figure.figsize'] = (8, 4)

## 1. A signal is a vector

We sample a continuous-looking signal at finitely many points. The computer stores only the vector of samples.

In [ ]:
n = 128
t = np.linspace(0, 1, n, endpoint=False)
x = 1.2*np.cos(2*np.pi*3*t) + 0.7*np.sin(2*np.pi*8*t)

print('Shape of x:', x.shape)
print('First 10 entries:', np.round(x[:10], 3))

plt.plot(t, x, marker='o', markersize=2)
plt.xlabel('time')
plt.ylabel('signal value')
plt.title('A sampled signal')
plt.grid(True, alpha=0.3)
plt.show()

**Reflection.** Which part is the mathematical vector? Which part is only a visualization choice?

## 2. Build wave vectors

A wave sampled at $n$ time points is also a vector in $\mathbb{R}^n$.

In [ ]:
n = 128
j = np.arange(n)
t = j/n

waves = {}
for f in [1, 3, 8, 20]:
    waves[f] = np.cos(2*np.pi*f*t)

for f, w in waves.items():
    plt.plot(t, w, label=f'f={f}')
plt.xlabel('time')
plt.ylabel('value')
plt.title('Sampled cosine wave vectors')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Orthogonality of different frequencies

Different Fourier wave directions are orthogonal when sampled in the correct periodic way.

In [ ]:
def cosine_wave(f, n=128):
    j = np.arange(n)
    return np.cos(2*np.pi*f*j/n)

freqs = np.arange(1, 13)
G = np.zeros((len(freqs), len(freqs)))
for a, fa in enumerate(freqs):
    for b, fb in enumerate(freqs):
        G[a, b] = np.dot(cosine_wave(fa), cosine_wave(fb))

plt.imshow(G)
plt.colorbar(label='dot product')
plt.xticks(range(len(freqs)), freqs)
plt.yticks(range(len(freqs)), freqs)
plt.xlabel('frequency')
plt.ylabel('frequency')
plt.title('Dot products between cosine wave vectors')
plt.show()

print(np.round(G, 2))

The diagonal entries are large because each wave has nonzero length. The off-diagonal entries are close to zero because different frequencies are orthogonal.

## 4. Fourier coefficients by projection

Before using FFT, we can manually project a signal onto sine and cosine wave directions.

In [ ]:
n = 256
j = np.arange(n)
t = j/n
x = 2.0*np.cos(2*np.pi*5*t) + 0.8*np.sin(2*np.pi*12*t)

cos_coeffs = []
sin_coeffs = []
for k in range(1, 31):
    ck = np.cos(2*np.pi*k*j/n)
    sk = np.sin(2*np.pi*k*j/n)
    cos_coeffs.append(2*np.dot(x, ck)/n)
    sin_coeffs.append(2*np.dot(x, sk)/n)

plt.stem(range(1,31), np.abs(cos_coeffs), linefmt='C0-', markerfmt='C0o', basefmt=' ', label='cosine')
plt.stem(range(1,31), np.abs(sin_coeffs), linefmt='C1-', markerfmt='C1s', basefmt=' ', label='sine')
plt.xlabel('frequency')
plt.ylabel('coefficient magnitude')
plt.title('Manual projection coefficients')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

**Student task.** Change the hidden frequencies and amplitudes in `x`. Verify that the plot finds them.

## 5. The DFT matrix

The discrete Fourier transform is a matrix multiplication. Its entries are complex waves.

In [ ]:
n = 8
j = np.arange(n)
k = j.reshape((n, 1))
F = np.exp(-2j*np.pi*k*j/n)

# Check orthogonality of rows: F F^* = n I
check = F @ F.conj().T
print(np.round(check, 6))

The Fourier matrix is orthogonal/unitary up to scaling. This is why Fourier coordinates can be inverted exactly.

## 6. FFT: finding frequencies quickly

Now we use `np.fft.fft`, which computes the same DFT much faster.

In [ ]:
n = 512
t = np.linspace(0, 1, n, endpoint=False)
x = 1.5*np.cos(2*np.pi*6*t) + 0.6*np.sin(2*np.pi*18*t) + 0.2*rng.normal(size=n)

X = np.fft.fft(x)
freq = np.fft.fftfreq(n, d=1/n)
positive = freq >= 0

plt.plot(t, x)
plt.title('Noisy signal')
plt.xlabel('time')
plt.grid(True, alpha=0.3)
plt.show()

plt.plot(freq[positive], np.abs(X[positive])/n)
plt.xlim(0, 50)
plt.xlabel('frequency')
plt.ylabel('magnitude')
plt.title('FFT magnitude spectrum')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Low-pass filtering

In frequency coordinates, we can remove high-frequency components. This is coordinate editing.

In [ ]:
n = 512
t = np.linspace(0, 1, n, endpoint=False)
clean = np.sin(2*np.pi*3*t) + 0.5*np.sin(2*np.pi*7*t)
noisy = clean + 0.6*rng.normal(size=n)

Y = np.fft.fft(noisy)
freq = np.fft.fftfreq(n, d=1/n)
cutoff = 10
mask = np.abs(freq) <= cutoff
filtered = np.fft.ifft(Y * mask).real

plt.plot(t, noisy, alpha=0.35, label='noisy')
plt.plot(t, clean, linewidth=2, label='clean')
plt.plot(t, filtered, linewidth=2, label='filtered')
plt.legend()
plt.title('Low-pass Fourier filtering')
plt.xlabel('time')
plt.grid(True, alpha=0.3)
plt.show()

print('Noisy RMSE:', np.sqrt(np.mean((noisy-clean)**2)))
print('Filtered RMSE:', np.sqrt(np.mean((filtered-clean)**2)))

**Student task.** Try different cutoff values. What happens if the cutoff is too small? Too large?

## 8. High-pass filtering and edge emphasis

High-frequency components often represent sharp changes. Let us build a signal with a jump.

In [ ]:
n = 512
t = np.linspace(0, 1, n, endpoint=False)
x = np.where(t < 0.45, 0.0, 1.0) + 0.1*np.sin(2*np.pi*5*t)
X = np.fft.fft(x)
freq = np.fft.fftfreq(n, d=1/n)

high = np.fft.ifft(X * (np.abs(freq) >= 15)).real
low = np.fft.ifft(X * (np.abs(freq) <= 15)).real

plt.plot(t, x, label='original')
plt.plot(t, low, label='low-pass')
plt.plot(t, high, label='high-pass')
plt.legend()
plt.title('Low-pass versus high-pass filtering')
plt.xlabel('time')
plt.grid(True, alpha=0.3)
plt.show()

## 9. Compression by keeping large coefficients

Instead of keeping low frequencies, we can keep the largest Fourier coefficients.

In [ ]:
n = 512
t = np.linspace(0, 1, n, endpoint=False)
x = np.sin(2*np.pi*3*t) + 0.5*np.sin(2*np.pi*9*t) + 0.25*np.sign(np.sin(2*np.pi*2*t))
X = np.fft.fft(x)

for keep in [6, 20, 60]:
    idx = np.argsort(np.abs(X))[-keep:]
    Xk = np.zeros_like(X)
    Xk[idx] = X[idx]
    xk = np.fft.ifft(Xk).real
    rmse = np.sqrt(np.mean((x - xk)**2))
    plt.plot(t, x, label='original')
    plt.plot(t, xk, label=f'keep {keep}, RMSE={rmse:.3f}')
    plt.legend()
    plt.title('Fourier coefficient compression')
    plt.xlabel('time')
    plt.grid(True, alpha=0.3)
    plt.show()

## 10. Spectrogram intuition: frequency changes over time

A global Fourier transform tells us which frequencies appear overall. But sometimes frequencies change over time. A spectrogram computes local Fourier transforms in sliding windows.

In [ ]:
n = 1024
t = np.linspace(0, 1, n, endpoint=False)
x = np.zeros(n)
x[t < 0.33] = np.sin(2*np.pi*6*t[t < 0.33])
x[(t >= 0.33) & (t < 0.66)] = np.sin(2*np.pi*18*t[(t >= 0.33) & (t < 0.66)])
x[t >= 0.66] = np.sin(2*np.pi*35*t[t >= 0.66])

window = 128
step = 16
spec = []
centers = []
for start in range(0, n-window, step):
    segment = x[start:start+window] * np.hanning(window)
    S = np.abs(np.fft.rfft(segment))
    spec.append(S)
    centers.append((start + window/2)/n)
spec = np.array(spec).T
freqs = np.fft.rfftfreq(window, d=1/n)

plt.imshow(spec, aspect='auto', origin='lower', extent=[centers[0], centers[-1], freqs[0], freqs[-1]])
plt.ylim(0, 60)
plt.colorbar(label='local magnitude')
plt.xlabel('time')
plt.ylabel('frequency')
plt.title('Simple spectrogram')
plt.show()

## 11. Two-dimensional Fourier transform for images

Images have horizontal and vertical frequencies.

In [ ]:
m = 128
y, xgrid = np.mgrid[0:m, 0:m]
img = 0.5 + 0.25*np.sin(2*np.pi*xgrid/24) + 0.2*np.cos(2*np.pi*y/16)
img += (((xgrid-82)**2 + (y-48)**2) < 18**2)*0.5
img = np.clip(img, 0, 1)

F2 = np.fft.fftshift(np.fft.fft2(img))
mag = np.log1p(np.abs(F2))

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].imshow(img, cmap='gray')
ax[0].set_title('synthetic image')
ax[0].axis('off')
ax[1].imshow(mag, cmap='magma')
ax[1].set_title('log Fourier magnitude')
ax[1].axis('off')
plt.show()

## 12. Image low-pass filtering

A circular low-pass filter in the 2D Fourier domain smooths an image.

In [ ]:
F = np.fft.fftshift(np.fft.fft2(img))
cy, cx = m//2, m//2
Y, Xg = np.ogrid[:m, :m]
r = np.sqrt((Y-cy)**2 + (Xg-cx)**2)

for radius in [8, 16, 32]:
    mask = r <= radius
    filtered = np.fft.ifft2(np.fft.ifftshift(F*mask)).real
    plt.imshow(filtered, cmap='gray', vmin=0, vmax=1)
    plt.title(f'2D Fourier low-pass, radius={radius}')
    plt.axis('off')
    plt.show()

## 13. High-dimensional viewpoint

The Fourier transform is an invertible linear transformation on $\mathbb{C}^n$. For many structured signals, the Fourier coordinate vector is sparse or nearly sparse. This is why Fourier compression can work.

In [ ]:
n = 1024
t = np.linspace(0, 1, n, endpoint=False)
# A structured signal with only a few frequencies
x_struct = sum(a*np.sin(2*np.pi*f*t + p) for a, f, p in [(1.0, 4, 0.1), (0.7, 17, 1.0), (0.4, 80, -0.6)])
# A random signal
x_rand = rng.normal(size=n)

X_struct = np.abs(np.fft.fft(x_struct))
X_rand = np.abs(np.fft.fft(x_rand))

plt.semilogy(np.sort(X_struct)[::-1]/np.max(X_struct), label='structured signal')
plt.semilogy(np.sort(X_rand)[::-1]/np.max(X_rand), label='random signal')
plt.xlabel('coefficient rank')
plt.ylabel('relative magnitude, log scale')
plt.title('Fourier sparsity: structured versus random')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Final reflection

Answer these in your own words.

1. Why is Fourier analysis linear algebra?
2. What does a Fourier coefficient measure geometrically?
3. Why does low-pass filtering smooth a signal?
4. When would Fourier compression work well? When would it work poorly?
5. How are Fourier, PCA, and SVD similar? How are they different?